# M3 — Priority (review check)

**Approach:** priority *double-check* (contributor D). Exact priority is unreliable (~50% ceiling), so this model estimates the priority and is used to **flag** cases where the agent's priority is `FLAG_GAP` (=2) or more levels away. Inbound features only — no `sla_target_hours` (derived from priority) and no plan proxy. Chronological split; ordinal extras + flag rate. Saves `models/m3_priority_review.joblib`.

In [ ]:
import sys, os, warnings; warnings.filterwarnings("ignore")
_p = os.getcwd()
while _p != os.path.dirname(_p) and not os.path.exists(os.path.join(_p, "northstar_prep.py")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import numpy as np, pandas as pd
import northstar_prep as P

In [ ]:
df = P.load_clean()
FEATS, TARGET, FLAG_GAP = ["case_summary","case_type","channel"], "priority", 2
labels = P.PRIORITIES
tr, va, te = P.time_split(df)
pipe = P.make_clf([P.t_word(), P.t_cat(["case_type","channel"])]).fit(tr[FEATS], tr[TARGET])
pred = pipe.predict(te[FEATS])
base = df[TARGET].value_counts(normalize=True).iloc[0]
metrics = P.evaluate(te[TARGET], pred, labels, baseline=base)
metrics.update(P.ordinal_extras(te[TARGET], pred, labels))
rank = P.PRIORITY_RANK
gap = (te[TARGET].map(rank).to_numpy() - np.array([rank[p] for p in pred]))
metrics["flag_rate"] = round(float((np.abs(gap) >= FLAG_GAP).mean()), 3)
metrics

In [ ]:
proba = pipe.predict_proba(te[FEATS])
test = te[["case_id","case_summary","case_type","channel",TARGET]].copy().reset_index(drop=True)
test["predicted"], test["confidence"] = pred, proba.max(axis=1).round(3)
full = P.make_clf([P.t_word(), P.t_cat(["case_type","channel"])]).fit(df[FEATS], df[TARGET])
P.save_bundle("m3_priority_review", {
    "model": full, "target": TARGET, "features": FEATS, "classes": labels, "kind": "multiclass",
    "label": "Priority — review check", "metrics": metrics, "test": test,
    "report": P.class_report(te[TARGET], pred, labels), "confusion": P.conf_matrix(te[TARGET], pred, labels),
    "meta": {"contributor":"D", "approach":"priority double-check (ordinal, flag mismatches)",
             "split":"time", "flag_gap":FLAG_GAP, "priorities":labels,
             "inputs":"summary + case_type + channel (no sla_target, no plan proxy)"}})